<a href="https://colab.research.google.com/github/andreas824/MLCB_team_project/blob/main/notebooks/reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase A — Reproduction & Preprocessing

Load the combined Maitra et al. (2023) snRNA-seq matrix (GSE213982 + GSE144136,
71 donors, ~160k nuclei), attach the authors' cell-type annotations, normalize,
and confirm the paper's Mic1 / ExN10_L46 signal as a sanity check. Output: a
clean AnnData checkpoint that feeds Phase B (CellChat).

**Runtime:** `Runtime > Change runtime type > CPU, High-RAM`. Phase A needs RAM,
not GPU.

Reusable functions live in `src/functions.py` (pulled from GitHub by the boot
cell). Data lives on Google Drive, never in git.

**Two run modes:**
- *First run:* execute every cell top to bottom (download → load → normalize →
  DEG → checkpoint).
- *Resume:* run the boot cell, then the Fast Resume cell, then jump to whatever
  you're working on — skips the slow download/load/normalize.

## 1. Environment

In [ ]:
# Install packages not preinstalled on Colab
!pip install -q scanpy anndata harmonypy leidenalg python-igraph GEOparse

import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import os

sc.settings.verbosity = 1
print("scanpy", sc.__version__)
print("anndata", ad.__version__)

## 2. Boot — mount Drive, clone repo, import functions

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

# --- data paths (Google Drive) ---
PROJECT_ROOT   = '/content/drive/MyDrive/MLCB_team_project'
RAW_DIR        = os.path.join(PROJECT_ROOT, 'data', 'raw')
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'data', 'checkpoints')
for d in (RAW_DIR, CHECKPOINT_DIR):
    os.makedirs(d, exist_ok=True)

# --- code (GitHub) ---
REPO_URL = 'https://github.com/andreas824/MLCB_team_project.git'
REPO_DIR = '/content/MLCB_team_project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
from functions import build_condition_map, load_dataset, normalize, deg_sanity_check

print("Raw data  ->", RAW_DIR)
print("Checkpoints ->", CHECKPOINT_DIR)

## 2b. Fast resume (optional)

After the first full run produced the final checkpoint, use this to jump
straight back to a ready, normalized `adata`. Run the boot cell above first,
then this cell, then go to whatever you're working on. **Skip this on the
first full run.**

In [ ]:
ckpt_final = os.path.join(CHECKPOINT_DIR, 'phaseA_final_for_cellchat.h5ad')
if os.path.exists(ckpt_final):
    adata = sc.read_h5ad(ckpt_final)
    print("Resumed from final checkpoint:", adata.shape)
    print(".X max:", round(float(adata.X.max()), 3), "(small = normalized+log)")
    print("counts layer:", 'counts' in adata.layers)
else:
    print("No final checkpoint yet — run the full pipeline below first.")

## 3. Download the combined matrix from GEO

Three supplementary files: the counts matrix (~1.1 GB, genes x cells), the cell
metadata (the `x` column encoding `donor.barcode.broad.fine`), and the gene names.
`wget -c` resumes on interruption, so a Colab timeout never restarts the download
from zero. Already-downloaded files are skipped.

In [ ]:
import subprocess

GSE = 'GSE213982'
gse_dir = os.path.join(RAW_DIR, GSE)
os.makedirs(gse_dir, exist_ok=True)

base = f"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE213nnn/{GSE}/suppl/"
files = [
    f"{GSE}_combined_counts_matrix.mtx.gz",
    f"{GSE}_combined_counts_matrix_cells_columns.csv.gz",
    f"{GSE}_combined_counts_matrix_genes_rows.csv.gz",
]
for fn in files:
    out = os.path.join(gse_dir, fn)
    if not os.path.exists(out) or os.path.getsize(out) == 0:
        print(f"Downloading {fn} ...")
        r = subprocess.run(["wget", "-c", "-O", out, base + fn])
        if r.returncode != 0:
            print(f"  WARNING: wget returned {r.returncode} for {fn}")
    print(f"  {fn}: {os.path.getsize(out)/1e6:.1f} MB")

## 4. Fetch SOFT metadata for the condition map

The cell string encodes donor / cell-type but **not** diagnosis (MDD vs Control).
Diagnosis lives in the GEO SOFT metadata: GSE213982 for females, GSE144136 for
males. These are small (~KB) — only metadata, not the male count matrix (already
inside the combined matrix above). Note: 38 female + 34 male GSM entries = 72,
but M24 appears twice (technical replicate M24_2), so there are 71 donors.

In [ ]:
import GEOparse

gse_female = GEOparse.get_GEO(geo='GSE213982',
                              destdir=os.path.join(RAW_DIR, 'GSE213982'), silent=True)
gse_male   = GEOparse.get_GEO(geo='GSE144136',
                              destdir=os.path.join(RAW_DIR, 'GSE144136'), silent=True)
print("Female GSMs:", len(gse_female.gsms))
print("Male GSMs:  ", len(gse_male.gsms))

## 5. Load + verify

`build_condition_map` reads diagnosis for all 71 donors. `load_dataset` loads the
matrix (streaming read to fit free-Colab RAM, transposed to cells x genes), parses
the cell string into metadata columns, merges the `M24_2` technical replicate onto
`M24`, and attaches sex/condition. Raw counts are preserved in `.layers['counts']`.

The crosstab is the key sanity check — it must match the paper:
**female 20 MDD / 18 Control, male 17 MDD / 16 Control**.

In [ ]:
condition_map = build_condition_map(gse_female, gse_male)
print(f"Condition map: {len(condition_map)} donors (expect 71)")

adata = load_dataset(RAW_DIR, condition_map)
print(adata)

# --- sanity checks ---
print("\nCells:", adata.n_obs, "| Genes:", adata.n_vars)
print("Unique donors:", adata.obs['donor_id'].nunique(), "(expect 71)")

donor_tbl = adata.obs[['donor_id', 'sex', 'condition']].drop_duplicates()
print("\nDonor-level condition x sex (expect F: 20/18, M: 17/16):")
print(pd.crosstab(donor_tbl['sex'], donor_tbl['condition']))

print("\nM24 merged correctly? (M24_2 gone):",
      'M24_2' not in adata.obs['donor_id'].unique())
print("Broad cell types:", sorted(adata.obs['cell_type_broad'].unique()))
print("N fine clusters:", adata.obs['cell_type_fine'].nunique())

## 6. Normalization

The authors already did QC and annotation (160,711 high-quality nuclei come
pre-labelled), so the only preprocessing step we add is normalization:
library-size scaling to counts-per-1e4, then log1p. This transforms `.X`;
raw counts stay safe in `.layers['counts']` for the downstream pseudobulk
(which needs raw counts, not normalized values).

In [ ]:
# Sanity: confirm .X currently holds raw integer counts before we transform
print("Before normalize:")
print("  .X dtype:", adata.X.dtype, "| max value:", adata.X.max())
print("  counts layer present:", 'counts' in adata.layers)

adata = normalize(adata)

# Confirm: .X is now normalized+log, raw counts safe in the layer
print("\nAfter normalize:")
print("  .X max value:", round(float(adata.X.max()), 3), "(log-scale, should be small)")
print("  layers['counts'] max:", adata.layers['counts'].max(), "(raw, unchanged)")
print("  .X is normalized+log, .layers['counts'] holds raw counts")

## 7. DEG sanity check — reproduce the Mic1 / ExN10_L46 signal

Per-cluster Wilcoxon (MDD vs Control), within one sex, on the authors' annotations.
This confirms the paper's central cell-type-specific finding exists in our data:
strong differential signal in **Mic1 for females** and **ExN10_L46 for males**.

**Caveat (for the report):** this is a *nucleus-level* test, so it suffers from
pseudoreplication — nuclei from the same donor are not independent, which inflates
the *number* of significant DEGs. The *direction and identity* of the top signal is
reliable (we verify ExN10_L46 at donor level below), but the definitive donor-level
statistics are deferred to the pseudobulk stage. This is exactly why the authors
used pseudobulk (muscat/edgeR) rather than nucleus-level testing.

In [ ]:
import matplotlib.pyplot as plt

deg_results = deg_sanity_check(adata)

# Quick visual: top DEG effect sizes per target
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, ((cluster, sex), df) in zip(axes, deg_results.items()):
    if df is None:
        ax.set_title(f"{cluster} ({sex}) — skipped")
        continue
    top = df.head(15).iloc[::-1]   # reverse so strongest is at top
    colors = ['firebrick' if lfc > 0 else 'steelblue'
              for lfc in top['logfoldchanges']]
    ax.barh(top['names'], top['logfoldchanges'], color=colors)
    ax.axvline(0, color='black', lw=0.5)
    ax.set_title(f"{cluster} ({sex})\nMDD vs Control, top 15 by padj")
    ax.set_xlabel('log fold change (MDD vs Control)')
plt.tight_layout()
plt.show()

## 8. Diagnostics — is the ExN10_L46 signal real?

The ExN10_L46 top hits include metabolic/housekeeping genes (GAPDH, PKM, CKB).
We check this isn't a technical artifact: (1) no large sequencing-depth difference
between MDD and Control nuclei, (2) no single donor dominating, and (3) the signal
holds at donor level (median expression per donor). All three confirm the signal
is real — consistent with known metabolic/synaptic dysregulation in deep-layer
excitatory neurons in MDD.

In [ ]:
# Depth + donor-balance checks
sub = adata[(adata.obs['cell_type_fine'] == 'ExN10_L46') &
            (adata.obs['sex'] == 'male')].copy()

sub.obs['total_counts'] = np.asarray(sub.layers['counts'].sum(axis=1)).ravel()
sub.obs['n_genes'] = np.asarray((sub.layers['counts'] > 0).sum(axis=1)).ravel()
print("Median total counts / genes per nucleus (MDD vs Control):")
print(sub.obs.groupby('condition', observed=True)[['total_counts', 'n_genes']].median())

print("\nNuclei per donor (top contributors):")
print(sub.obs.groupby(['condition', 'donor_id'], observed=True).size()
      .sort_values(ascending=False).head(10))

print("\nDonors with nuclei in ExN10_L46 per condition:")
print(sub.obs.groupby('condition', observed=True)['donor_id'].nunique())

In [ ]:
# Donor-level confirmation: do top genes hold when aggregated per donor?
check_genes = ['GAPDH', 'PKM', 'CKB', 'VAMP2', 'SNCB']
gene_idx = [sub.var_names.get_loc(g) for g in check_genes if g in sub.var_names]

donor_means = {}
for donor in sub.obs['donor_id'].unique():
    dmask = (sub.obs['donor_id'] == donor).values
    cond = sub.obs.loc[dmask, 'condition'].iloc[0]
    means = np.asarray(sub.X[dmask][:, gene_idx].mean(axis=0)).ravel()
    donor_means[donor] = (cond, means)

dm = pd.DataFrame({d: m for d, (c, m) in donor_means.items()}, index=check_genes).T
dm['condition'] = [c for d, (c, m) in donor_means.items()]
print("Per-donor mean expression by condition (median across donors):")
print(dm.groupby('condition', observed=True).median())

## 9. Final checkpoint — handoff to CellChat (Phase B)

Save the clean, normalized, fully annotated AnnData. This is the single Phase A
deliverable: `phaseA_final_for_cellchat.h5ad`. The Fast Resume cell at the top
reloads exactly this.

For Phase B: run CellChat on the **7 broad cell classes** (enough nuclei per donor
for reliable ligand-receptor inference); the 41 fine clusters are also present in
case finer resolution of the microglia↔neuron axis is needed.

In [ ]:
ckpt_final = os.path.join(CHECKPOINT_DIR, 'phaseA_final_for_cellchat.h5ad')
adata.write_h5ad(ckpt_final)

print("=== Phase A deliverable ===")
print("Saved:", ckpt_final, f"({os.path.getsize(ckpt_final)/1e6:.1f} MB)")
print("\nContents for Phase B (CellChat):")
print(f"  cells: {adata.n_obs} | genes: {adata.n_vars} | donors: {adata.obs['donor_id'].nunique()}")
print(f"  .X = normalized+log | .layers['counts'] = raw")
print(f"  obs columns: {list(adata.obs.columns)}")
print(f"  broad cell types ({adata.obs['cell_type_broad'].nunique()}):",
      sorted(adata.obs['cell_type_broad'].unique()))
print(f"  donors per (sex x condition) — the 4 CellChat groups:")
print(adata.obs.groupby(['sex', 'condition'], observed=True)['donor_id'].nunique())